In [ ]:
from pathlib import Path
from typing import Sequence
import random
import time

import chipwhisperer as cw


Q = 127
N = 16
K = 8
ROW_LENGTH = N - K

FQ_ELEM_BYTES = 1
ROW_BYTES = ROW_LENGTH * FQ_ELEM_BYTES

INPUT_BYTES = 2 * ROW_BYTES
OUTPUT_BYTES = ROW_BYTES

SUBROUTINE_ROW_MUL3 = 0x03
BAUDRATE = 230400

FIRMWARE_DIRECTORY = Path.cwd()

HEX_BINARY = (
    FIRMWARE_DIRECTORY
    / "less_row_mul3-PINATA.hex"
)

print("Row length  :", ROW_LENGTH)
print("Input bytes :", INPUT_BYTES)
print("Output bytes:", OUTPUT_BYTES)
print("HEX file    :", HEX_BINARY)

In [ ]:
def _validate_row(
    row: Sequence[int],
    name: str,
) -> None:
    if len(row) != ROW_LENGTH:
        raise ValueError(
            f"Expected {ROW_LENGTH} entries in {name}, "
            f"got {len(row)}"
        )

    if any(
        value < 0 or value >= Q
        for value in row
    ):
        raise ValueError(
            f"{name} entries must be in [0, {Q - 1}]"
        )


def serialize_row_mul3(
    in1: Sequence[int],
    in2: Sequence[int],
) -> bytes:
    """Serialize two LESS rows in firmware order."""

    _validate_row(in1, "in1")
    _validate_row(in2, "in2")

    payload = (
        bytes(in1)
        + bytes(in2)
    )

    if len(payload) != INPUT_BYTES:
        raise RuntimeError(
            f"Serialized payload has {len(payload)} bytes, "
            f"expected {INPUT_BYTES}"
        )

    return payload


def row_mul3_python(
    in1: Sequence[int],
    in2: Sequence[int],
) -> list[int]:
    """Independent Python calculation for row_mul3."""

    _validate_row(in1, "in1")
    _validate_row(in2, "in2")

    return [
        (left * right) % Q
        for left, right in zip(in1, in2)
    ]

In [ ]:
def connect_and_flash():
    """Connect, configure the clock and flash the firmware."""

    if not HEX_BINARY.is_file():
        raise FileNotFoundError(
            f"Firmware image not found: {HEX_BINARY}"
        )

    scope = cw.scope()
    scope.default_setup()

    scope.io.clkout = 15e6

    print("Clock output:", scope.io.clkout)

    time.sleep(0.5)

    cw.program_target(
        scope,
        cw.programmers.STM32FProgrammer,
        str(HEX_BINARY),
        baud=BAUDRATE,
    )

    target = cw.target(
        scope,
        cw.targets.SimpleSerial2,
    )

    target.baud = BAUDRATE
    target.flush()

    return scope, target


def upload_row_mul3_input(
    target,
    payload: bytes,
    verify_readback: bool = True,
) -> None:
    """Upload the complete 16-byte row_mul3 input."""

    if len(payload) != INPUT_BYTES:
        raise ValueError(
            f"Expected {INPUT_BYTES} input bytes, "
            f"got {len(payload)}"
        )

    subcommand = (
        0x80
        if verify_readback
        else 0x00
    )

    target.send_cmd(
        "s",
        subcommand,
        bytearray(payload),
    )

    if verify_readback:
        readback = target.simpleserial_read(
            "r",
            INPUT_BYTES,
            timeout=1000,
            ack=True,
        )

        if readback is None:
            raise TimeoutError(
                "No row_mul3 input readback received"
            )

        if bytes(readback) != payload:
            raise RuntimeError(
                "row_mul3 input readback mismatch"
            )

    else:
        acknowledgement = (
            target.simpleserial_wait_ack(
                timeout=1000
            )
        )

        if acknowledgement is None:
            raise TimeoutError(
                "No acknowledgement for row_mul3 input"
            )


def row_mul3_board(
    target,
    in1: Sequence[int],
    in2: Sequence[int],
) -> list[int]:
    """Execute row_mul3 on the Pinata."""

    payload = serialize_row_mul3(
        in1,
        in2,
    )

    upload_row_mul3_input(
        target,
        payload,
        verify_readback=True,
    )

    target.send_cmd(
        "b",
        SUBROUTINE_ROW_MUL3,
        bytearray(),
    )

    response = target.simpleserial_read(
        "r",
        OUTPUT_BYTES,
        timeout=1000,
        ack=True,
    )

    if response is None:
        raise TimeoutError(
            "No result received from row_mul3"
        )

    if len(response) != OUTPUT_BYTES:
        raise RuntimeError(
            f"Expected {OUTPUT_BYTES} result bytes, "
            f"received {len(response)}"
        )

    return list(response)

In [ ]:
scope = None
target = None

try:
    scope, target = connect_and_flash()

    deterministic_in1 = list(
        range(1, ROW_LENGTH + 1)
    )

    deterministic_in2 = list(
        reversed(deterministic_in1)
    )

    expected = row_mul3_python(
        deterministic_in1,
        deterministic_in2,
    )

    actual = row_mul3_board(
        target,
        deterministic_in1,
        deterministic_in2,
    )

    print("Input 1 :", deterministic_in1)
    print("Input 2 :", deterministic_in2)
    print("Expected:", expected)
    print("Board   :", actual)

    if actual != expected:
        raise AssertionError(
            "Pinata result does not match "
            "the Python result"
        )

    number_of_random_tests = 100

    for test_index in range(
        number_of_random_tests
    ):
        in1 = [
            random.randrange(Q)
            for _ in range(ROW_LENGTH)
        ]

        in2 = [
            random.randrange(Q)
            for _ in range(ROW_LENGTH)
        ]

        expected = row_mul3_python(
            in1,
            in2,
        )

        actual = row_mul3_board(
            target,
            in1,
            in2,
        )

        if actual != expected:
            raise AssertionError(
                f"Mismatch in random test {test_index}: "
                f"expected {expected}, got {actual}"
            )

    print(
        f"OK: deterministic test and "
        f"{number_of_random_tests} random tests passed."
    )

finally:
    if target is not None:
        target.dis()

    if scope is not None:
        scope.dis()